# 6. What does a detector measure from one diffraction order?

[Notebook 5](../5_Grating_Diffraction_in_Three_Dimensions.ipynb) established which order directions are allowed and which a pupil collects. Here we select **one propagating order** and connect its field to the brightness recorded by a detector.

Imagine isolating that order where the outgoing beams are separated, then observing a broad part of the selected beam. We model this region as a uniform plane wave, ignoring beam edges. Its direction is fixed; its amplitude is supplied as an input. Calculating the amplitude produced by a particular grating is a separate task.

The question for this lesson is: **why can the field change sign, phase, and instantaneous value while the recorded brightness stays the same?**

Our route is **field at one point → averaged detector signal → one order across the detector**. We use the amplitude and phase notation from [Notebook 0](../0_Foundations_of_Scalar_Optics.ipynb) and [Notebook 1](../1_Plane_Waves_and_Wavevectors.ipynb).

Run top to bottom with NumPy, Matplotlib, Plotly, and ipywidgets. The three explorers and two small calculations have independent settings. Field units and plot scales stay fixed. We retain the monochromatic scalar model, a homogeneous lossless medium, and the $e^{-i\omega t}$ convention.

In [ ]:
# Find the shared wave_optics package in the project root.
from pathlib import Path
import sys
import importlib

_support_dir = next((base
    for base in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (base / 'wave_optics' / '__init__.py').is_file()), None)
if _support_dir is None:
    raise FileNotFoundError('Open this notebook within the Grating_Physics folder.')
if str(_support_dir) not in sys.path:
    sys.path.insert(0, str(_support_dir))
# Relocate a cached package without changing the lesson's module reload order.
importlib.invalidate_caches()
if 'wave_optics' in sys.modules:
    sys.modules['wave_optics'].__path__[:] = [str(_support_dir / 'wave_optics')]

import importlib
import numpy as np
from IPython.display import display

# Rerunning setup releases any earlier copies of this notebook's views.
if 'amplitude_lessons' in globals():
    for old_lesson in amplitude_lessons.values():
        old_lesson.close()

# Refresh shared helpers before the views that import them.
for _module_name in (
    'wave_optics._view_cache',
    'wave_optics.diffraction',
    'wave_optics.plotting',
    'wave_optics.grating_intro',
    'wave_optics.waves',
    'wave_optics.amplitude_view',
):
    if _module_name in sys.modules:
        importlib.reload(sys.modules[_module_name])
    else:
        importlib.import_module(_module_name)

import wave_optics.amplitude_view as amplitude_view
amplitude_lessons = {}

def show_amplitude_lesson(stage):
    if stage in amplitude_lessons:
        amplitude_lessons[stage].close()
    lesson = amplitude_view.AmplitudeIntensityExplorer(stage)
    amplitude_lessons[stage] = lesson
    display(lesson.ui)

## 1. The field at one observation point

For our selected order,
$$u_m(\mathbf r,t)=A_m\cos(\mathbf k_m\cdot\mathbf r+\phi_m-\omega t).$$
At a fixed point $P$, write $A=A_m$ and $\phi=\mathbf k_m\cdot\mathbf r_P+\phi_m$:
$$u(P,t)=A\cos(\phi-\omega t),\qquad U(P)=Ae^{i\phi}.$$

Here $A\geq0$ sets the extrema $\pm A$; $\phi$ sets the cycle's alignment with the time origin. The real field can be negative or cross zero while its amplitude stays nonzero.

An ordinary optical detector averages over many carrier cycles. The two panels below distinguish the **signed field** from its **square**. For one complete period $T$,
$$\langle u\rangle_T=0,\qquad
\langle u^2\rangle_T=A^2\langle\cos^2(\phi-\omega t)\rangle_T=\frac{A^2}{2}.$$
Angle brackets mean a time average. Squaring first prevents the positive and negative field readings from cancelling. A phase shift moves the cycle without changing its mean square.

**Try:** start with $A=1$, phase $0^\circ$. Change only phase to $90^\circ$, then change amplitude to 2. Which average changes? The horizontal axis is model time $t/T$; these curves are not camera frames.

In [ ]:
show_amplitude_lesson('average')


<details><summary>What to observe</summary>

At $A=1$, the mean square is $1/2$ for either phase, even though the field at $t=0$ changes from 1 to 0. At $A=2$, the mean square becomes 2. The mean signed field stays zero in every case.

</details>

## 2. Convert the mean square into recorded intensity

For this travelling wave at fixed medium and detector geometry,
$$I=K\langle u^2\rangle_T=K\frac{A^2}{2}.$$
The positive constant $K$ accounts for the physical field units and the fixed geometry. Compare with a reference wave in the same direction and medium, with amplitude $A_{\rm ref}=1$ field unit:
$$\boxed{\frac{I}{I_{\rm ref}}
=\frac{KA^2/2}{KA_{\rm ref}^2/2}
=\frac{A^2}{A_{\rm ref}^2}
=\frac{|U|^2}{A_{\rm ref}^2}.}$$

This is the squared-magnitude rule previewed in Notebook 0. The factor $1/2$ cancels in the ratio. Our plots show dimensionless **relative intensity**; absolute intensity in $\mathrm{W/m^2}$ requires the physical conversion.

**Predict and try:** change amplitude from 1 to 0.5, then to 2. The relative intensities should be 1, 0.25, and 4. The reference and the graph scales remain fixed.

In [ ]:
show_amplitude_lesson('intensity')


### Small calculation 1: can the starting field predict brightness?

The code compares three phases at the same amplitude. **Predict:** which rows have the same intensity? Then change `A` from 1 to 2 and rerun. These values are independent of the sliders.

`np.mean(u**2)` averages squared field samples over one cycle, without duplicating the endpoint.

In [ ]:
A = 1.0  # field units; try 2.0
A_ref = 1.0  # keep this reference fixed
phase_deg = np.array([0.0, 90.0, 180.0])
tau = np.linspace(0.0, 1.0, 400, endpoint=False)
reference_mean_square = A_ref**2 / 2
print('phase (degrees)    u(0)    mean(u²)    I/I_ref')
for phase in phase_deg:
    u = A * np.cos(np.deg2rad(phase) - 2 * np.pi * tau)
    mean_square = np.mean(u**2)
    relative_intensity = mean_square / reference_mean_square
    print(f'{phase:8.0f}          {u[0]:6.3f}     {mean_square:6.3f}      {relative_intensity:6.3f}')

<details><summary>Answer</summary>

At $A=1$, the starting fields are $1,0,-1$, but every row has mean square $0.5$ and relative intensity 1. At $A=2$, the starting fields double and every relative intensity becomes 4. One instantaneous field reading does not determine brightness.

</details>

### Read the measurement in reverse

For background-subtracted counts $S$ from a linear, unsaturated detector, keep the sampled area, exposure, wavelength, detector settings, and beam geometry the same:
$$\frac{S}{S_{\rm ref}}=\frac{I}{I_{\rm ref}}
\quad\Longrightarrow\quad
\frac{A}{A_{\rm ref}}=\sqrt{\frac{S}{S_{\rm ref}}}.$$
A change from 100 to 400 counts means four times the intensity and twice the field amplitude. These counts give no phase offset: all three phases in the calculation produce the same signal.

## 3. Compare a field snapshot with the detector image

Return to the same selected order across space. Use the earlier geometry: wavelength $\lambda=2\,\mu\mathrm m$ in the medium, grating period $d=4\,\mu\mathrm m$, normal incidence, and $m=+1$ travelling into $z<0$. Its fixed direction is
$$\frac{\mathbf k_m}{k}=\left(\frac12,0,-\frac{\sqrt3}{2}\right),
\qquad k=\frac{2\pi}{\lambda}.$$
Give it an amplitude $A_m$ and a phase offset $\phi_m$ at the spatial origin:
$$U_m(\mathbf r)=A_m e^{i(\mathbf k_m\cdot\mathbf r+\phi_m)}.$$
Different positions have different local phases, but
$$\boxed{\frac{I_m(\mathbf r)}{I_{\rm ref}}
=\frac{|U_m(\mathbf r)|^2}{A_{\rm ref}^2}
=\frac{A_m^2}{A_{\rm ref}^2},}$$
which is uniform.

The **left panel** shows the signed field in an $xz$ slice at the selected model time. The cyan line marks the detector plane at $z=-1.5\,\mu\mathrm m$. The **right panel** views that detector face-on in $xy$ and shows its ideal averaged intensity. Both panels cover a finite viewing window of a broad plane wave.

The detector reference has the same incidence direction, so the fixed projection onto the detector plane cancels in the intensity ratio. The grayscale stays at 0–4; it is not rescaled separately for each image.

**Try, one change at a time:**

1. Start with $A_m=1$, phase $0^\circ$, and $t/T=0$. Compare the alternating field colours with the uniform detector image.
2. Change phase to $90^\circ$, then model time to $0.25$. Do either of these changes affect the detector brightness?
3. Increase amplitude to 2, then set it to 0. Does the detector show stripes at either setting?

The time control selects an instantaneous field snapshot; it does not change the detector exposure.

In [ ]:
show_amplitude_lesson('order')


<details><summary>What to observe</summary>

Phase and time move the signed-field bands while the detector image stays uniform at relative intensity 1. Amplitude 2 gives intensity 4 everywhere; amplitude 0 gives zero everywhere. At zero amplitude the order direction remains geometrically allowed, but this component has no field and its phase is undefined.

The red and blue bands are signs of the field, not bright and dark optical fringes.

</details>

### Small calculation 2: sample three detector positions

Evaluate the complex field at three points on the same detector plane. **Predict:** can their instantaneous readings differ while their intensities agree? Start with `A_m = 0.5`, then change it to 1. The written settings are independent of the explorer.

In [ ]:
wavelength = 2.0  # micrometres; fixed selected-order geometry
k = 2 * np.pi / wavelength
k_m = k * np.array([0.5, 0.0, -np.sqrt(3) / 2])
A_m, A_ref = 0.5, 1.0  # try A_m = 1.0; keep the reference fixed
phase_deg, time_over_T = 0.0, 0.0

points = np.array([[-1.0, 0.0, -1.5],
                   [0.0, 0.0, -1.5],
                   [1.0, 0.0, -1.5]])  # micrometres
U_m = A_m * np.exp(1j * (points @ k_m + np.deg2rad(phase_deg)))
u_m = np.real(U_m * np.exp(-2j * np.pi * time_over_T))
I_relative = np.abs(U_m)**2 / A_ref**2

print('x (um)   instantaneous u   I_m/I_ref')
print(np.round(np.column_stack((points[:, 0], u_m, I_relative)), 3))

<details><summary>Answer</summary>

At $A_m=0.5$, the three instantaneous readings differ, while every relative intensity is 0.25. At $A_m=1$, the readings double and every intensity becomes 1. Changing phase or model time changes the instantaneous readings but preserves these intensities.

</details>

## What this establishes for Notebook 7

For one selected plane-wave order:

- Field amplitude sets recorded intensity through $I_m\propto A_m^2$.
- A constant phase shift leaves that order's intensity unchanged.
- Spatial bands in a signed-field snapshot are not intensity fringes.

We have supplied $A_m$ to study measurement. Determining its value from the grating structure requires a diffraction-amplitude calculation; $I_m/I_{\rm ref}$ here is not an order efficiency or a fraction of incident power.

[Notebook 7.1](../7_1_Interference_and_Mutual_Coherence_of_Two_Plane_Waves.ipynb) adds a second field at the same detector point. Its starting question is: **if phase does not affect either wave's intensity alone, why can their relative phase affect the intensity when they overlap?**

## References and practical notes

- [Notebook 0](../0_Foundations_of_Scalar_Optics.ipynb) and [Notebook 1](../1_Plane_Waves_and_Wavevectors.ipynb): complex-field notation, phase, and wavevectors.
- [OpenStax, *University Physics*, §16.3](https://openstax.org/books/university-physics-volume-2/pages/16-3-energy-carried-by-electromagnetic-waves), Eqs. (16.30)–(16.31): cycle averaging and the quadratic relation between travelling-wave field amplitude and intensity.
- Goodman, *Introduction to Fourier Optics*, second edition, §4.1.1, Eq. (4-5), printed p. 65: scalar squared-magnitude intensity convention.

No earlier notebook must be executed. Rerunning setup closes earlier views; rerunning a display cell replaces that view. Saved outputs and widget state are clear. To release all views when finished:

```python
for lesson in amplitude_lessons.values():
    lesson.close()
```